In [1]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
MODEL_TYPE = "DeepCNNLSTM"  

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs\\ablations_hpo"
FILE_NAME = f"ablation_A12_1s3w_hpo_v1"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: ablation_A12_1s3w_hpo_v1
From path: C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\ablations_hpo\ablation_A12_1s3w_hpo_v1.log


In [2]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'ablation_A12_1s3w_hpo_v1'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 37 trials.
Best value (Accuracy): 0.7334
Best Hyperparameters:
  outer_lr: 0.00012394663013612985
  wd: 0.0001221028322177272
  maml_inner_steps: 7
  maml_alpha_init: 0.006085033056507095
  maml_alpha_init_eval: 0.005739307922339232
  maml_use_lslr: True
  use_lslr_at_eval: False
  use_maml_msl: hybrid
  maml_msl_num_epochs: 32
  episodes_per_epoch_train: 500
  num_experts: 26
  MOE_top_k: 8
  MOE_gate_temperature: 1.1772057737569557
  MOE_aux_coeff: 0.19978032332648069
  MOE_ctx_out_dim: 16
  MOE_ctx_hidden_dim: 32
  MOE_dropout: 0.0057674320648040366
  MOE_aux_loss_plcmt: both


In [4]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [5]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [6]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [7]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [8]:
# FULL

#params_to_plot_ARCH = [
#    "cnn_base_filters", "lstm_hidden", "groupnorm_num_groups", "use_GlobalAvgPooling"
#]

params_to_plot_OPT = [
    "outer_lr", "wd", #"label_smooth", "ft_lr"
]

params_to_plot_MAML = [
    "maml_inner_steps", "maml_alpha_init_eval", "maml_use_lslr", "use_lslr_at_eval", "use_maml_msl", "episodes_per_epoch_train"
]

params_to_plot_MOE_CORE = [
    "num_experts", "MOE_top_k", "MOE_aux_coeff", "MOE_gate_temperature"
]

params_to_plot_MOE_AUX = [
    "MOE_ctx_out_dim", "MOE_ctx_hidden_dim", "MOE_dropout"
]

In [9]:
from optuna.visualization import plot_slice


In [10]:
fig_slice = plot_slice(study, params=params_to_plot_OPT)
fig_slice.show()

In [11]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_CORE)
fig_slice.show()

In [12]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_AUX)
fig_slice.show()

In [13]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
16,16,0.733438,None,2026-04-20 07:26:16.633128,0 days 03:40:28.535432
9,9,0.675417,None,2026-04-20 06:50:34.464122,0 days 02:05:43.279502
19,19,0.651979,None,2026-04-20 07:57:05.111093,0 days 03:35:47.610533
6,6,0.642083,None,2026-04-20 06:13:59.318856,0 days 04:24:21.833018
7,7,0.632188,None,2026-04-20 06:16:58.980387,0 days 01:49:32.216072
22,22,0.628437,None,2026-04-20 08:07:09.229127,0 days 02:34:09.066685
27,27,0.570312,None,2026-04-20 09:09:57.852061,0 days 02:55:39.451685
3,3,0.518958,None,2026-04-20 06:11:00.359187,0 days 01:45:23.718841
26,26,0.505833,None,2026-04-20 08:56:47.527172,0 days 03:07:06.833222
21,21,0.450833,None,2026-04-20 08:06:30.267178,0 days 00:37:06.074427


In [ ]:
from optuna.trial import TrialState

# 1. Filter for only successfully completed trials
completed_trials = [t for t in study.trials if t.state == TrialState.COMPLETE]

# 2. Sort the filtered list (using reverse=True if you want the highest values)
top_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:10]

# 3. Print the params
for t in top_trials:
    print(t.params)